# Attenuator Calibration Lab Script

Dirty science-lab notebook for attenuator calibration. It has two paths:

- embedded auto-calibration: run firmware, retrieve retained HAC4 metadata and raw record chunks, and inspect them with helper plots;
- manual exploration: call `pcb.atten()`, wait, and read `pcb.pd()` so thresholds and failure modes can be changed quickly without reflashing.

Measure dark in a preceding cell or in the dark cell below before collecting either dataset.
The embedded firmware is the source of truth; the manual path below is intentionally written to mirror the current C state machine.



In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
import math
import time
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np

try:
    from scipy.optimize import least_squares
except ImportError:
    least_squares = None

import hispec_fibpcb as hspcb

BROKER = "hispec.caltech.edu"
LASER = "1028y"
OUTPUT = "yj_ao"
FIBER = "M"
PD_CHANNEL = "yj"

TAKE_DARK = False
DARK_DURATION_MS = 2000

MANUAL_FIT_METHOD: Literal["firmware", "scipy"] = "firmware"
MANUAL_DWELL_S = 0.55
MANUAL_DWELL_MS = int(round(MANUAL_DWELL_S * 1000))
MANUAL_SWEEP_STEP_MV = 50.0
MANUAL_COMPANION_SEARCH_MIN_STEP_MV = 5.0
MANUAL_COMPANION_SEARCH_MAX_TRIES = 16
MANUAL_MAX_RECORDS = 128
MANUAL_INITIAL_LASER_LEVELS_PCT = (100.0, 50.0, 5.0)
MANUAL_SNR_USABLE = hspcb.ATTENUATOR_CAL_SNR_USABLE
MANUAL_ADC_LSB_MV = 0.1875
MANUAL_DAC_SIGMA_MV = 3.0
MANUAL_FIT_MAX_ITER = 30
MANUAL_FIT_INITIAL_LAMBDA = 1.0e-3
MANUAL_FIT_MIN_SLOPE = 1.0e-12
MANUAL_FIT_MAX_SLOPE = 1.0
MANUAL_RECORDS_PER_CHUNK = 37
TX_MIN = hspcb.ATTEN_CAL_MIN_TX
TX_MAX = hspcb.ATTEN_CAL_MAX_TX
MAX_DAC_MV = hspcb.ATTENUATOR_DRIVE_MAX_MV
ADC_CLIP_MV = hspcb.ATTENUATOR_ADC_CLIP_MV
GAIN = hspcb.ATTENUATOR_DEFAULT_GAIN

pcb = hspcb.HispecFibPcb(BROKER, connect=True)
pcb.status(), pcb.time()



## Dark

Dark is deliberately outside attenuator calibration. Take or force the channel dark here, then leave calibration to read the photodiode configurable window.



In [ ]:
pcb.pd(PD_CHANNEL)



In [ ]:
if TAKE_DARK:
    pcb.pd_dark(PD_CHANNEL, duration_ms=DARK_DURATION_MS, persist=False)
pcb.pd(PD_CHANNEL)



## Embedded Firmware Path

Run the on-board algorithm, then fetch metadata plus fixed raw-record chunks. The Python helper derives bridge scale, scaled signal, transmission, dB attenuation, fit inclusion, residuals, and role markers from those records.



In [ ]:
embedded_status = pcb.atten_calibrate_auto(
    LASER,
    output=OUTPUT,
    fiber=FIBER,
    dwell_ms=MANUAL_DWELL_MS,
    persist=False,
)
embedded_status



In [ ]:
pcb.atten_calibration_status()



In [ ]:
embedded = pcb.atten_calibration_data(physical="all")
embedded



In [ ]:
embedded.bridge_table()



In [ ]:
embedded.plot_physical("dac1")
embedded.plot_physical("dac2")



In [ ]:
coeff = pcb.atten_coeff(LASER)
embedded.plot_surface(coeff.dac1, coeff.dac2, overlay_records=True)



## Manual Exploration Path

The manual path mirrors the current C algorithm:

1. For each physical FVOA, hold the DUT at 0 mV and binary-search the companion FVOA for the lowest usable companion DAC. Lower companion DAC means more light; higher companion DAC means more attenuation.
2. The selected usable `initial_probe` record is the open reference. There is no synthetic `reference` record and no repeated confirmation measurement.
3. Sweep the DUT linearly from 0 mV to `MAX_DAC_MV` using `MANUAL_SWEEP_STEP_MV`. Saturated sweep points are retained and skipped forward; below-SNR points trigger bridge normalization.
4. A bridge uses the latest usable retained `point` in the current segment as the before side, increments the segment, searches the companion FVOA, and stores the accepted `bridge_probe` record as the after side in the bridge table.
5. Fitting is done after acquisition in dB output space from raw records plus metadata. The firmware-style fit is a local finite-difference Gauss-Newton loop; the SciPy fit uses the same residuals and propagated dB uncertainty.



In [ ]:
@dataclass
class CalRecord:
    physical: str
    event: Literal["point", "initial_probe", "bridge_probe"]
    classification: Literal["ok", "saturated", "below_snr", "adc_error"]
    segment: int
    sweep_mv: float
    other_mv: float
    laser_pct: float
    signal_mv: float
    signal_err_mv: float
    max_mv: float

    @property
    def usable(self) -> bool:
        return self.classification == "ok" and self.signal_mv > 0.0 and self.signal_err_mv > 0.0

    @property
    def acq_snr(self) -> float:
        return self.signal_mv / self.signal_err_mv if self.signal_err_mv > 0.0 else np.nan


@dataclass
class BridgeRole:
    before_record: int
    after_record: int


@dataclass
class CompanionSearch:
    low_mv: float
    high_mv: float
    candidate_record: int | None = None
    tries: int = 0

    def midpoint(self) -> float:
        return 0.5 * (self.low_mv + self.high_mv)

    def done(self) -> bool:
        return (
            self.high_mv - self.low_mv <= MANUAL_COMPANION_SEARCH_MIN_STEP_MV
            or self.tries >= MANUAL_COMPANION_SEARCH_MAX_TRIES
        )

    def note(self, rec: CalRecord, record_index: int) -> None:
        if rec.classification == "saturated":
            self.low_mv = rec.other_mv
        elif rec.classification == "ok":
            self.high_mv = rec.other_mv
            self.candidate_record = record_index
        elif rec.classification == "below_snr":
            self.high_mv = rec.other_mv
        else:
            raise RuntimeError(f"companion search failed at {rec.other_mv:.2f} mV: {rec.classification}")
        self.tries += 1


@dataclass
class CalRun:
    physical: Literal["dac1", "dac2"]
    records: list[CalRecord]
    reference_record: int | None = None
    bridges: list[BridgeRole] | None = None

    def __post_init__(self) -> None:
        if self.bridges is None:
            self.bridges = []

    def append(self, rec: CalRecord) -> int:
        index = len(self.records)
        self.records.append(rec)
        return index


def _record_rows(run: CalRun) -> list[tuple]:
    rows = []
    for i, rec in enumerate(run.records):
        rows.append(
            (
                rec.physical,
                i,
                rec.event,
                rec.classification,
                int(rec.segment),
                float(rec.sweep_mv),
                float(rec.sweep_mv) * GAIN,
                float(rec.other_mv),
                float(rec.other_mv) * GAIN,
                float(rec.laser_pct),
                float(rec.signal_mv),
                float(rec.signal_err_mv),
                float(rec.max_mv),
            )
        )
    return rows


def _run_metadata(run: CalRun) -> dict:
    chunk_count = int(math.ceil(len(run.records) / MANUAL_RECORDS_PER_CHUNK)) if run.records else 0
    return {
        "state": "complete",
        "mode": "manual",
        "physical": run.physical,
        "fit_valid": False,
        "fit_accepted": False,
        "record_overflow": len(run.records) >= MANUAL_MAX_RECORDS,
        "record_size": hspcb._ATTEN_CAL_RECORD_BINARY.size,
        "records_per_chunk": MANUAL_RECORDS_PER_CHUNK,
        "record_count": len(run.records),
        "record_chunk_count": chunk_count,
        "reference_valid": run.reference_record is not None,
        "reference_record": int(run.reference_record or 0),
        "bridge_count": len(run.bridges or ()),
        "bridges": tuple(
            {
                "bridge_index": i,
                "before_record": int(bridge.before_record),
                "after_record": int(bridge.after_record),
            }
            for i, bridge in enumerate(run.bridges or ())
        ),
    }


def run_to_dataset(run: CalRun) -> hspcb.AttenuatorCalibrationDataset:
    records = np.array(_record_rows(run), dtype=hspcb.ATTEN_CAL_DTYPE).view(np.recarray)
    return hspcb.AttenuatorCalibrationDataset(records=records, meta=(_run_metadata(run),))


def runs_to_dataset(*runs: CalRun) -> hspcb.AttenuatorCalibrationDataset:
    rows = []
    metas = []
    for run in runs:
        rows.extend(_record_rows(run))
        metas.append(_run_metadata(run))
    records = np.array(rows, dtype=hspcb.ATTEN_CAL_DTYPE).view(np.recarray)
    return hspcb.AttenuatorCalibrationDataset(records=records, meta=tuple(metas))



In [ ]:
def _set_pair(physical: str, sweep_mv: float, other_mv: float):
    sweep_mv = float(np.clip(sweep_mv, 0.0, MAX_DAC_MV))
    other_mv = float(np.clip(other_mv, 0.0, MAX_DAC_MV))
    if physical == "dac1":
        return pcb.atten(LASER, value1_mv=sweep_mv, value2_mv=other_mv)
    if physical == "dac2":
        return pcb.atten(LASER, value1_mv=other_mv, value2_mv=sweep_mv)
    raise ValueError("physical must be dac1 or dac2")


def _pd_window(channel: str = PD_CHANNEL):
    pd = pcb.pd(channel)
    pd_channel = getattr(pd, channel)
    if pd_channel is None:
        raise RuntimeError(f"pd/{channel} response did not include {channel}")
    return pd_channel.window


def _classify_window(window) -> tuple[str, float, float, float, float]:
    signal_mv = float(window.mean_net_mv)
    signal_err_mv = float(window.mean_net_err_mv)
    max_mv = float(window.max_mv)
    if not (signal_err_mv > 0.0 and np.isfinite(signal_err_mv)):
        signal_err_mv = MANUAL_ADC_LSB_MV
    if signal_mv >= ADC_CLIP_MV:
        return "saturated", signal_mv, signal_err_mv, max_mv, np.nan
    snr = signal_mv / signal_err_mv
    if signal_mv <= 0.0 or not np.isfinite(snr) or snr < MANUAL_SNR_USABLE:
        return "below_snr", signal_mv, signal_err_mv, max_mv, snr
    return "ok", signal_mv, signal_err_mv, max_mv, snr


def _print_record(rec: CalRecord) -> None:
    print(
        f"{rec.event:14s} {rec.physical}={rec.sweep_mv:8.2f} other={rec.other_mv:8.2f} "
        f"signal={rec.signal_mv:9.3f}+/-{rec.signal_err_mv:6.3f} "
        f"snr={rec.acq_snr:8.2f} classification={rec.classification:10s} segment={rec.segment}"
    )


def measure_point(
    run: CalRun,
    sweep_mv: float,
    other_mv: float,
    *,
    event: Literal["point", "initial_probe", "bridge_probe"],
    segment: int,
    laser_pct: float,
) -> tuple[int, CalRecord]:
    _set_pair(run.physical, sweep_mv, other_mv)
    time.sleep(MANUAL_DWELL_S)
    classification, signal_mv, signal_err_mv, max_mv, _snr = _classify_window(_pd_window(PD_CHANNEL))
    rec = CalRecord(
        physical=run.physical,
        event=event,
        classification=classification,
        segment=int(segment),
        sweep_mv=float(sweep_mv),
        other_mv=float(other_mv),
        laser_pct=float(laser_pct),
        signal_mv=signal_mv,
        signal_err_mv=signal_err_mv,
        max_mv=max_mv,
    )
    index = run.append(rec)
    _print_record(rec)
    return index, rec


def _next_linear_sweep_mv(sweep_mv: float, step_mv: float) -> float:
    return min(MAX_DAC_MV, float(sweep_mv) + float(step_mv))



In [ ]:
def _initial_reference(run: CalRun, laser_levels_pct: tuple[float, ...]) -> tuple[float, float]:
    laser_index = 0
    laser_pct = float(laser_levels_pct[laser_index])
    pcb.set_laser_level(LASER, laser_pct)
    search = CompanionSearch(0.0, MAX_DAC_MV)

    while len(run.records) < MANUAL_MAX_RECORDS:
        index, rec = measure_point(
            run,
            0.0,
            search.midpoint(),
            event="initial_probe",
            segment=0,
            laser_pct=laser_pct,
        )
        search.note(rec, index)

        if (
            rec.classification == "saturated"
            and rec.other_mv >= MAX_DAC_MV - MANUAL_COMPANION_SEARCH_MIN_STEP_MV
        ):
            laser_index += 1
            if laser_index >= len(laser_levels_pct):
                raise RuntimeError(f"{run.physical} initial probe saturated at max companion attenuation")
            laser_pct = float(laser_levels_pct[laser_index])
            pcb.set_laser_level(LASER, laser_pct)
            search = CompanionSearch(0.0, MAX_DAC_MV)
            continue

        if search.done():
            if search.candidate_record is None:
                raise RuntimeError(f"{run.physical} initial companion search found no usable reference")
            run.reference_record = search.candidate_record
            return run.records[search.candidate_record].other_mv, laser_pct

    raise RuntimeError(f"{run.physical} exceeded manual record budget during initial search")


def _latest_bridge_anchor(run: CalRun, segment: int) -> tuple[int | None, CalRecord | None, bool]:
    count = 0
    all_below_snr = True
    for index in range(len(run.records) - 1, -1, -1):
        rec = run.records[index]
        if rec.segment == segment and rec.event == "point":
            count += 1
            if rec.classification != "below_snr":
                all_below_snr = False
            if rec.classification == "ok":
                return index, rec, False
    if all_below_snr and count == 1 and segment > 0:
        return None, None, True
    raise RuntimeError(f"{run.physical} has no usable bridge anchor in segment {segment}")


def _bridge_normalize(
    run: CalRun,
    *,
    segment: int,
    other_mv: float,
    step_mv: float,
    laser_pct: float,
) -> tuple[bool, int, float, float]:
    if other_mv <= MANUAL_COMPANION_SEARCH_MIN_STEP_MV:
        return True, segment, other_mv, MAX_DAC_MV

    before_index, anchor, done = _latest_bridge_anchor(run, segment)
    if done:
        return True, segment, other_mv, MAX_DAC_MV
    assert anchor is not None and before_index is not None

    segment += 1
    search = CompanionSearch(0.0, other_mv)
    while len(run.records) < MANUAL_MAX_RECORDS:
        index, rec = measure_point(
            run,
            anchor.sweep_mv,
            search.midpoint(),
            event="bridge_probe",
            segment=segment,
            laser_pct=laser_pct,
        )
        search.note(rec, index)

        if not search.done():
            continue

        if search.candidate_record is None:
            search_floor = 0.0
            for prior in reversed(run.records):
                if prior.event == "bridge_probe" and prior.segment == segment:
                    if prior.classification == "saturated":
                        search_floor = max(search_floor, prior.other_mv)
                        break
            if search_floor == 0.0:
                print(f"{run.physical} bridge probes were all below SNR; finishing physical")
                return True, segment, other_mv, MAX_DAC_MV
            if abs(search_floor - other_mv) < MANUAL_COMPANION_SEARCH_MIN_STEP_MV:
                raise RuntimeError(f"{run.physical} bridge search found no viable companion point")
            search = CompanionSearch(search_floor, other_mv)
            continue

        after_index = search.candidate_record
        before = run.records[before_index]
        after = run.records[after_index]
        ratio = after.signal_mv / before.signal_mv if before.signal_mv > 0.0 else np.nan
        if not (ratio > 1.0 and np.isfinite(ratio)):
            raise RuntimeError(f"{run.physical} bridge ratio is invalid: {ratio}")
        run.bridges.append(BridgeRole(before_record=before_index, after_record=after_index))
        next_sweep = _next_linear_sweep_mv(anchor.sweep_mv, step_mv)
        return False, segment, after.other_mv, next_sweep

    raise RuntimeError(f"{run.physical} exceeded manual record budget during bridge search")


def acquire_physical(
    physical: Literal["dac1", "dac2"],
    *,
    step_mv: float = MANUAL_SWEEP_STEP_MV,
    laser_levels_pct: tuple[float, ...] = MANUAL_INITIAL_LASER_LEVELS_PCT,
) -> CalRun:
    run = CalRun(physical=physical, records=[])
    segment = 0
    sweep_mv = 0.0
    other_mv, laser_pct = _initial_reference(run, laser_levels_pct)

    while len(run.records) < MANUAL_MAX_RECORDS:
        _index, rec = measure_point(
            run,
            sweep_mv,
            other_mv,
            event="point",
            segment=segment,
            laser_pct=laser_pct,
        )

        if rec.classification in ("ok", "saturated"):
            if sweep_mv >= MAX_DAC_MV:
                break
            sweep_mv = _next_linear_sweep_mv(sweep_mv, step_mv)
            continue

        if rec.classification == "below_snr":
            if sweep_mv >= MAX_DAC_MV:
                break
            done, segment, other_mv, sweep_mv = _bridge_normalize(
                run,
                segment=segment,
                other_mv=other_mv,
                step_mv=step_mv,
                laser_pct=laser_pct,
            )
            if done:
                break
            continue

        raise RuntimeError(f"{physical} sweep failed: {rec.classification}")

    return run



In [ ]:
def _manual_fit_points(run: CalRun):
    derived = run_to_dataset(run).derived()
    mask = np.asarray(derived.fit_candidate, dtype=bool) & np.isfinite(derived.db) & np.isfinite(derived.db_err)
    points = derived[mask].view(np.recarray)
    if len(points) < 6:
        raise RuntimeError(f"not enough fit records: {len(points)}")
    return points


def _manual_model_db(params, dac_mv, *, gain: float = GAIN):
    return hspcb._atten_db_from_coeff((float(params[0]), float(params[1]), gain), dac_mv)


def _manual_model_d_db_d_dac(params, dac_mv, *, gain: float = GAIN):
    step = max(MANUAL_DAC_SIGMA_MV, 0.5)
    dac = np.asarray(dac_mv, dtype=float)
    lo = np.clip(dac - step, 0.0, MAX_DAC_MV)
    hi = np.clip(dac + step, 0.0, MAX_DAC_MV)
    span = np.maximum(hi - lo, 1.0e-12)
    return (_manual_model_db(params, hi, gain=gain) - _manual_model_db(params, lo, gain=gain)) / span


def _manual_weighted_residual(params, points, *, gain: float = GAIN):
    model = _manual_model_db(params, points.sweep_mv, gain=gain)
    d_db_d_dac = _manual_model_d_db_d_dac(params, points.sweep_mv, gain=gain)
    sigma = np.sqrt(points.db_err * points.db_err + (d_db_d_dac * MANUAL_DAC_SIGMA_MV) ** 2)
    return (model - points.db) / np.maximum(sigma, hspcb.ATTEN_CAL_MIN_DB_ERR)


def _manual_initial_guess(points, *, gain: float = GAIN):
    x = np.asarray(points.sweep_mv, dtype=float) * gain
    db = np.asarray(points.db, dtype=float)
    f50 = float(x[np.nanargmin(np.abs(db - 3.01029995664))])
    span = float(np.nanmax(x) - np.nanmin(x))
    slope = 8.0 / span if span > 0.0 else 8.0 / (MAX_DAC_MV * gain)
    max_fvoa_mv = MAX_DAC_MV * gain
    return np.array([
        np.clip(f50, 1.0, 2.0 * max_fvoa_mv),
        np.clip(slope, MANUAL_FIT_MIN_SLOPE, MANUAL_FIT_MAX_SLOPE),
    ])


def _fit_records_firmware_style(run: CalRun, *, gain: float = GAIN):
    points = _manual_fit_points(run)
    params = _manual_initial_guess(points, gain=gain)
    cost = float(np.sum(_manual_weighted_residual(params, points, gain=gain) ** 2))
    lam = MANUAL_FIT_INITIAL_LAMBDA
    max_fvoa_mv = MAX_DAC_MV * gain
    for _ in range(MANUAL_FIT_MAX_ITER):
        r = _manual_weighted_residual(params, points, gain=gain)
        step_f50 = max(abs(params[0]) * 1.0e-5, 0.1)
        step_slope = max(abs(params[1]) * 1.0e-5, 1.0e-8)
        r_f50 = _manual_weighted_residual(params + np.array([step_f50, 0.0]), points, gain=gain)
        r_slope = _manual_weighted_residual(params + np.array([0.0, step_slope]), points, gain=gain)
        j0 = (r_f50 - r) / step_f50
        j1 = (r_slope - r) / step_slope
        h00 = float(np.dot(j0, j0) + lam)
        h01 = float(np.dot(j0, j1))
        h11 = float(np.dot(j1, j1) + lam)
        g0 = float(np.dot(j0, r))
        g1 = float(np.dot(j1, r))
        det = h00 * h11 - h01 * h01
        if not (det > 0.0 and np.isfinite(det)):
            raise RuntimeError("firmware-style fit became singular")
        d_f50 = (-h11 * g0 + h01 * g1) / det
        d_slope = (h01 * g0 - h00 * g1) / det
        trial = np.array([
            np.clip(params[0] + d_f50, 1.0, 2.0 * max_fvoa_mv),
            np.clip(params[1] + d_slope, MANUAL_FIT_MIN_SLOPE, MANUAL_FIT_MAX_SLOPE),
        ])
        trial_cost = float(np.sum(_manual_weighted_residual(trial, points, gain=gain) ** 2))
        if np.isfinite(trial_cost) and trial_cost < cost:
            if abs(trial[0] - params[0]) < 1.0e-6 and abs(trial[1] - params[1]) < 1.0e-12:
                params = trial
                cost = trial_cost
                break
            params = trial
            cost = trial_cost
            lam = max(lam * 0.3, 1.0e-12)
        else:
            lam = min(lam * 10.0, 1.0e12)
    return _manual_fit_summary("firmware", run, points, params, gain=gain)


def _fit_records_scipy(run: CalRun, *, gain: float = GAIN):
    if least_squares is None:
        raise RuntimeError("scipy is not installed")
    points = _manual_fit_points(run)
    initial = _manual_initial_guess(points, gain=gain)
    result = least_squares(
        lambda p: _manual_weighted_residual(p, points, gain=gain),
        initial,
        bounds=([1.0, MANUAL_FIT_MIN_SLOPE], [2.0 * MAX_DAC_MV * gain, MANUAL_FIT_MAX_SLOPE]),
        loss="linear",
        max_nfev=5000,
    )
    return _manual_fit_summary("scipy", run, points, result.x, gain=gain, success=bool(result.success))


def _manual_fit_summary(method: str, run: CalRun, points, params, *, gain: float = GAIN, success: bool = True):
    model = _manual_model_db(params, points.sweep_mv, gain=gain)
    residual = model - points.db
    return {
        "method": method,
        "physical": run.physical,
        "accepted": bool(success and params[0] > 0.0 and params[1] > 0.0),
        "points": int(len(points)),
        "fvoa_50pct_mv": float(params[0]),
        "slope_inv_fvoa_mv": float(params[1]),
        "gain": float(gain),
        "rms_db": float(np.sqrt(np.mean(residual * residual))),
        "max_abs_db": float(np.max(np.abs(residual))),
        "min_tx": float(np.nanmin(points.tx)),
        "max_tx": float(np.nanmax(points.tx)),
    }


def fit_records(run: CalRun, *, method: str = MANUAL_FIT_METHOD, gain: float = GAIN):
    if method == "firmware":
        return _fit_records_firmware_style(run, gain=gain)
    if method == "scipy":
        return _fit_records_scipy(run, gain=gain)
    raise ValueError("method must be 'firmware' or 'scipy'")



In [ ]:
def plot_manual_records(run: CalRun, fit: dict[str, float | int | bool]):
    coeff = (float(fit["fvoa_50pct_mv"]), float(fit["slope_inv_fvoa_mv"]), float(fit["gain"]))
    dataset = run_to_dataset(run)
    derived = dataset.derived(
        dac1=coeff if run.physical == "dac1" else None,
        dac2=coeff if run.physical == "dac2" else None,
    )
    rec = derived[np.asarray(derived.physical).astype(str) == run.physical].view(np.recarray)
    fvoa = rec.fvoa_mv
    events = np.asarray(rec.event).astype(str)
    classifications = np.asarray(rec.classification).astype(str)
    included = np.asarray(rec.included, dtype=bool)

    fig, axes = plt.subplots(4, 1, figsize=(9, 10), sharex=True, constrained_layout=True)
    fig.suptitle(f"{run.physical} manual calibration records")

    axes[0].set_title("raw dark-subtracted signal")
    finite_signal = np.isfinite(rec.signal_mv) & np.isfinite(rec.signal_err_mv)
    axes[0].errorbar(fvoa[finite_signal], rec.signal_mv[finite_signal], yerr=rec.signal_err_mv[finite_signal], fmt="none", ecolor="0.65", linewidth=0.7)
    axes[0].set_ylabel("signal_mv")

    axes[1].set_title("bridge-scaled signal")
    finite_scaled = np.isfinite(rec.scaled_signal_mv) & np.isfinite(rec.scaled_signal_err_mv)
    axes[1].errorbar(fvoa[finite_scaled], rec.scaled_signal_mv[finite_scaled], yerr=rec.scaled_signal_err_mv[finite_scaled], fmt="none", ecolor="0.65", linewidth=0.7)
    axes[1].set_ylabel("scaled_signal_mv")

    axes[2].set_title("attenuation from normalized transmission")
    finite_db = np.isfinite(rec.db) & np.isfinite(rec.db_err)
    axes[2].errorbar(fvoa[finite_db], rec.db[finite_db], yerr=rec.db_err[finite_db], fmt="none", ecolor="0.65", linewidth=0.7)
    grid_dac = np.linspace(0.0, MAX_DAC_MV, 400)
    axes[2].plot(grid_dac * coeff[2], hspcb._atten_db_from_coeff(coeff, grid_dac), color="black", linewidth=1.0, label=f"{fit['method']} fit")
    axes[2].set_ylabel("attenuation_db")
    axes[2].legend(loc="best")

    event_colors = {"point": "tab:blue", "initial_probe": "0.55", "bridge_probe": "tab:green"}
    event_markers = {"point": "o", "initial_probe": "s", "bridge_probe": "P"}
    for event in dict.fromkeys(events):
        mask = events == event
        color = event_colors.get(event, "0.35")
        marker = event_markers.get(event, "o")
        axes[0].scatter(fvoa[mask], rec.signal_mv[mask], marker=marker, color=color, s=32, alpha=0.82, label=event)
        axes[1].scatter(fvoa[mask], rec.scaled_signal_mv[mask], marker=marker, color=color, s=32, alpha=0.82)
        axes[2].scatter(fvoa[mask], rec.db[mask], marker=marker, color=color, s=32, alpha=0.82)

    role_styles = {
        "reference": ("*", "tab:cyan"),
        "bridge_before": ("^", "tab:orange"),
        "bridge_after": ("v", "tab:red"),
    }
    for role, indices in dataset._role_record_indices(run.physical).items():
        if not indices:
            continue
        mask = np.isin(np.asarray(rec.record, dtype=int), np.asarray(indices, dtype=int))
        marker, color = role_styles[role]
        for ax, y in zip(axes[:3], (rec.signal_mv, rec.scaled_signal_mv, rec.db), strict=False):
            finite = mask & np.isfinite(y)
            if np.any(finite):
                ax.scatter(fvoa[finite], y[finite], marker=marker, facecolors="none", edgecolors=color, s=92, linewidths=1.2, label=role)

    bad = classifications != "ok"
    axes[0].scatter(fvoa[bad], rec.signal_mv[bad], marker="x", color="black", s=38, label="not ok")
    axes[0].legend(loc="best", fontsize="small")

    axes[3].axhline(0.0, color="0.35", linestyle="--", linewidth=0.8)
    axes[3].scatter(fvoa[included], rec.residual_db[included], color="black", s=24)
    axes[3].set_ylabel("residual_db")
    axes[3].set_xlabel("swept FVOA drive (mV)")
    return fig



In [ ]:
dac1_run = acquire_physical("dac1", step_mv=MANUAL_SWEEP_STEP_MV)
dac1_fit = fit_records(dac1_run, method=MANUAL_FIT_METHOD)
dac1_fit



In [ ]:
plot_manual_records(dac1_run, dac1_fit)



In [ ]:
dac2_run = acquire_physical("dac2", step_mv=MANUAL_SWEEP_STEP_MV)
dac2_fit = fit_records(dac2_run, method=MANUAL_FIT_METHOD)
dac2_fit



In [ ]:
plot_manual_records(dac2_run, dac2_fit)



In [ ]:
manual_records = [asdict(record) for record in dac1_run.records + dac2_run.records]
manual_coeff = {
    "dac1": {k: dac1_fit[k] for k in ("fvoa_50pct_mv", "slope_inv_fvoa_mv", "gain")},
    "dac2": {k: dac2_fit[k] for k in ("fvoa_50pct_mv", "slope_inv_fvoa_mv", "gain")},
}
manual_dataset = runs_to_dataset(dac1_run, dac2_run)
manual_coeff



In [ ]:
manual_dataset.bridge_table()



In [ ]:
manual_dataset.plot_surface(manual_coeff["dac1"], manual_coeff["dac2"], overlay_records=True)



In [ ]:
# Review plots before applying. Keep persist=False until repeated and accepted.
# pcb.set_atten_coeff(LASER, manual_coeff["dac1"], manual_coeff["dac2"], persist=False)

